# s20 — Context Compression

**What this teaches:** how Yoke keeps a long-running session within the model's context window. The `compress` plugin tracks per-session token usage and fires two distinct triggers — a *soft* one that asks the model to summarise older turns into a memory file, and a *hard* one that force-truncates if the soft pass didn't free enough space. The same plugin also exposes a `compact_now` tool the agent can call deliberately.

**Concept anchor:** compression is just another participant in the loop. It observes events emitted by the runner, decides when to act, and writes audit traces to disk so you can see exactly *what* was compressed and *why*.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars before launching Jupyter, e.g.:
  ```
  export YOKE_PROVIDER=openai_compat
  export OPENAI_BASE_URL=http://localhost:11434/v1
  export YOKE_MODEL=qwen2.5-coder
  ```
  Defaults to a local Ollama. Swap in `anthropic` / `openai` / `gemini` for hosted providers — see [docs/providers.md](../../docs/providers.md).
- This notebook writes a compression audit log to `.agent_memory.md` in the current working directory.

## 1. Imports

Beyond `agentkit` and `stream`, we pull in `core/tools` for the file-system tool set and `internal/compress` for the compression plugin and its companion tools.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/yoke/core/agentkit"
    "github.com/blouargant/yoke/core/stream"
    fstools "github.com/blouargant/yoke/core/tools"
    "github.com/blouargant/yoke/internal/compress"
)

## 2. Helper

We swap the example's `os.Exit`-based `must` for a panic version so a notebook error doesn't kill the GoNB kernel.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Build the compress plugin

`compress.PluginWithTools` returns three things:

- A **plugin** value to attach to the runner — it subscribes to model/tool events to observe token usage.
- A **`compactTools`** slice — the `compact_now` tool the agent can invoke deliberately.
- A **`wait`** func — blocks until any in-flight compression goroutine has finished writing its audit file (handy at end-of-run).

In production the soft trigger fires near **75%** of the window. We intentionally shrink `WindowTokens` to 800 here and set `SoftRatio: 0.5`, `HardRatio: 0.8`, so a single demo prompt is enough to trip both triggers and you can see them in the audit file.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)
plug, compactTools, wait, err := compress.PluginWithTools("compress", compress.Config{
    WindowTokens: 800, // tiny so the demo triggers at all
    SoftRatio:    0.5,
    HardRatio:    0.8,
    AuditPath:    ".agent_memory.md",
    LLM:          llm,
})
must(err)
fmt.Println("compress plugin wired; audit will land in .agent_memory.md")

## 4. Wire the agent and runner

We hand the agent the default file-system tool set plus the `compactTools` returned above. The plugin is attached to the **runner**, not the agent — plugins live alongside the loop, not inside it.

In [ ]:
tools := append(fstools.New(), compactTools...)
a, err := agentkit.New(agentkit.AgentConfig{
    Name:  "s20_compress",
    Model: llm,
    Tools: tools,
})
must(err)
r, err := agentkit.Runner("s20", a, plug)
must(err)

## 5. Run a prompt that fills the window

Ask the model to generate enough text that the 800-token window gets blown out within one turn. After `RunOnce` returns we call `wait()` so the asynchronous compression pass has time to flush its audit log.

In [ ]:
prompt := "Write a 500-word essay about agent harnesses, then list 5 key terms."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))
wait()
fmt.Println("(see .agent_memory.md for the compression audit log)")

## What to look for

- Open `.agent_memory.md` in the working directory after the run — you'll see timestamped entries for each soft/hard trigger, the prompt that was summarised, and the resulting memory snippet that gets prepended to future turns.
- The plugin is asynchronous: it watches events emitted by the same bus used in [s18_events](../s18_events/s18_events.ipynb). Compression never blocks the model loop.
- Compare with [s14_cache](../s14_cache/s14_cache.ipynb), which observes the *same* loop to compute prompt-cache hit rates — both plugins prove plugins are passive observers, not interceptors.

## Try it yourself

1. Bump `WindowTokens` to `8000` and re-run the same prompt — the soft trigger likely won't fire, and `.agent_memory.md` stays empty.
2. Add the `compact_now` tool to the instruction (`"Call compact_now after every long answer."`) and watch the audit file fill up with *deliberate* compressions instead of triggered ones.